In [2]:
import numpy as np

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import SimpleRNN
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import TimeDistributed

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences



In [4]:

# Dataset

english_sentences = [
   "hello",
    "good morning",
    "good evening",
    "i am hungry",
    "i need water",
    "i love you",
    "the cat is sleeping",
    "learning ai is fun"
]

german_sentences = [
   "hallo",
    "guten morgen",
    "guten abend",
    "ich habe hunger",
    "ich brauche wasser",
    "ich liebe dich",
    "die katze schlaeft",
    "ki lernen macht spass"
]


In [5]:

# 2. Tokenization

eng_tokenizer = Tokenizer()

ger_tokenizer = Tokenizer()

eng_tokenizer.fit_on_texts(english_sentences)

ger_tokenizer.fit_on_texts(german_sentences)

X = eng_tokenizer.texts_to_sequences(english_sentences)

y = ger_tokenizer.texts_to_sequences(german_sentences)

In [8]:

# 3. Padding


max_len = max(
    max(len(seq) for seq in X),
    max(len(seq) for seq in y)
)

X = pad_sequences(
    X,
    maxlen=max_len,
    padding="post"
)

y = pad_sequences(
    y,
    maxlen=max_len,
    padding="post"
)

In [9]:
# -----------------------------------

# 4. Reshape output

# -----------------------------------

# Required for TimeDistributed

y = np.expand_dims(y, axis=-1)

In [10]:
# -----------------------------------

# 5. Vocabulary sizes

# -----------------------------------

eng_vocab_size = len(eng_tokenizer.word_index) + 1

ger_vocab_size = len(ger_tokenizer.word_index) + 1

In [ ]:

# 6. Build model

model = Sequential()

# Embedding layer

model.add(
    Embedding( input_dim=eng_vocab_size, output_dim=8,input_length=max_len
    )
)

In [ ]:
# RNN layer

# IMPORTANT:

# return_sequences=True

model.add(

    SimpleRNN(

        16,

        return_sequences=True

    )

)

# Output layer

# Predicts one German word at each timestep

model.add(

    TimeDistributed(

        Dense(

            ger_vocab_size,

            activation="softmax"

        )

    )

)

In [13]:


# -----------------------------------

# 7. Compile model

# -----------------------------------

model.compile(

    optimizer="adam",

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)



In [14]:
# -----------------------------------

# 8. Model summary

# -----------------------------------

model.summary()

# -----------------------------------

# 9. Train model

# -----------------------------------

model.fit(

    X,

    y,

    epochs=500,

    batch_size=2

)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ ?                      │   0 (unbuilt) │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.2812 - loss: 2.9218  
Epoch 2/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3750 - loss: 2.9029
Epoch 3/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3750 - loss: 2.8837
Epoch 4/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4062 - loss: 2.8641
Epoch 5/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4062 - loss: 2.8418
Epoch 6/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4062 - loss: 2.8203
Epoch 7/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4062 - loss: 2.7929
Epoch 8/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4062 - loss: 2.7625
Epoch 9/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4062 - loss: 2.7290
Epoch 10/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4062 - loss: 2.6967
Epoch 11/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3750 - loss: 2.6519
Epoch 12/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3750 - loss: 2.6044

In [16]:

print("\nTranslations:\n")

for sentence in english_sentences:

    # Convert sentence into numbers

    sequence = eng_tokenizer.texts_to_sequences([sentence])

    # Padding

    sequence = pad_sequences(

        sequence,

        maxlen=max_len,

        padding="post"

    )

    # Predict

    prediction = model.predict(sequence)

    # Get highest probability ids

    predicted_ids = np.argmax(

        prediction,

        axis=-1

    )

    # Convert ids back to words

    translated_sentence = ger_tokenizer.sequences_to_texts(

        predicted_ids

    )

    # Print result

    print(sentence, "→", translated_sentence[0])


Translations:

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
hello → hallo
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
good morning → guten morgen
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
good evening → guten abend
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
i am hungry → ich habe hunger
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
i need water → ich brauche wasser
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
i love you → ich liebe dich
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
the cat is sleeping → die katze schlaeft
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
learning ai is fun → ki lernen macht spass
